In [119]:
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import torch as th
from torch import nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo không cần thiết để Log sạch đẹp hơn

## [CẤU HÌNH CHIẾN LƯỢC ĐẦU TƯ - AI TRADING]

## Khối cấu hình trung tâm (Control Panel) giúp bạn tinh chỉnh "tính cách" của AI 

In [120]:
# mà không cần can thiệp sâu vào bên trong mã nguồn.
class CONFIG:
    # 1. PHÍ GIAO DỊCH (Transaction Cost): 
    # Tỷ lệ phần trăm CTCK thu cho mỗi lần bạn mua hoặc bán cổ phiếu. (VD: 0.2% = 0.002)
    COST_RATE = 0.0003
    
    # 2. CHU KỲ REBALANCE (Rule 1: Holding Period):
    # Thời gian (tính bằng ngày) AI bị ép phải "ôm chặt" danh mục trước khi được phép mua bán lại.
    # Mục đích: Giảm thiểu Over-trading (giao dịch quá nhiều) để không bị bào mòn tài khoản bởi phí giao dịch.
    HOLDING_PERIOD = 7
    
    # 3. NGƯỠNG TỰ TIN VÀO CỔ PHIẾU (Rule 5: Cash Conviction Threshold):
    # Mức độ tự tin tối thiểu (từ 0.0 đến 1.0) AI cần có để quyết định giải ngân.
    # Nếu mức tự tin cao nhất của AI thấp hơn ngưỡng này (thị trường đang quá rủi ro/khó đoán), 
    # nó sẽ tự động chừa lại phần vốn đó dưới dạng Tiền mặt để phòng thủ.
    CASH_CONVICTION_THRESHOLD = 0.2
    
    # 4. HÌNH PHẠT RỦI RO (Rule 3: Sharpe Penalty):
    # Mức điểm âm sẽ phạt AI nếu danh mục của nó chọn bị Lỗ sau 1 chu kỳ Holding Period.
    # Con số càng âm (VD: -2.0 hoặc -5.0) sẽ rèn luyện cho AI tính cẩn thận, chỉ mua mã an toàn.
    PENALTY_FOR_LOSS = -1.0
    
    # 5. CẮT LỖ ĐỘNG (Rule 6: Stop-Loss):
    # Mức rớt giá tối đa cho phép. Nếu đang trong thời gian "ôm chặt" mà danh mục sập chạm mốc này,
    # lệnh ngắt mạch sẽ kích hoạt: Bán tháo toàn bộ danh mục ngay lập tức để rút về Tiền mặt bảo toàn tính mạng.
    STOP_LOSS_THRESHOLD = -0.07
    
    # 6. CHỐT LỜI SỚM (Rule 6: Take-Profit):
    # Mức lãi kỳ vọng. Nếu danh mục sinh lời nhanh và đạt mốc này trước khi hết hạn ôm,
    # kích hoạt ngắt mạch: Chốt lời toàn bộ, bỏ túi tiền mặt và đứng ngoài thị trường chờ chu kỳ mới.
    TAKE_PROFIT_THRESHOLD = 0.15
    
    # 7. CHẾ ĐỘ CHỜ HÀNG VỀ T+2 (Rule 7: Settlement Lock):
    # Đặc sản của Chứng khoán VN. Số ngày cổ phiếu bị "nhốt" chưa về tài khoản. 
    # Trong những ngày này (VD: 2 ngày đầu tiên), AI tuyệt đối không thể bán cắt lỗ/chốt lời dù thị trường sập.
    T_PLUS_SETTLEMENT = 3
    
    # 8. CHIA KHÚC DỮ LIỆU ĐỂ HỌC VÀ THI (Rule 4: Walk-Forward Folds):
    # Kỹ thuật chống "Học Vẹt" (Overfitting). Dữ liệu sẽ được chia làm N phần. 
    # AI sẽ học ở quá khứ và bị quăng vào tương lai xa lạ để thi đấu.
    WALK_FORWARD_SPLITS = 3
    
    # 8. SỐ BƯỚC ĐÀO TẠO (Training Timesteps):
    # Số vòng lặp để AI cập nhật bộ não. Càng cao (50k - 100k) AI càng thông minh nhưng thời gian train càng lâu.
    TRAINING_TIMESTEPS = 500000
    
    # 9. ĐỘ PHÂN BỔ VỐN (Entropy Coefficient):
    # Quyết định AI sẽ All-in hay Rải rác.
    # Giá trị nhỏ (0.001) -> Ưu tiên All-in vài mã mạnh nhất.
    # Giá trị lớn (0.05) -> Ưu tiên chia đều tiền ra mua nhiều mã để phân tán rủi ro.
    ENT_COEF = 0.001

    # Neuron Network

## LUẬT 2: MARKET CONTEXT (TỔNG HỢP VĨ MÔ & VI MÔ)

## Hàm này nạp dữ liệu từ file Parquet (đã chạy HMM) và chế biến thêm các chỉ báo để tạo thành 

In [ ]:
def load_data():
    # 1. Nạp dữ liệu thô
    raw_df = pd.read_parquet(r"C:\Users\ADMIN\Desktop\Kaggle\output\hmm_v3_op1_extended\master_drl_ready_full.parquet")
    raw_df['time'] = pd.to_datetime(raw_df['time'])
    
    # Lấy các DataFrame cơ sở
    returns_df = raw_df.pivot(index='time', columns='ticker', values='log_return').fillna(0)
    weights_dim = returns_df.shape[1] # Số lượng mã cổ phiếu
    
    vol_df = raw_df.pivot(index='time', columns='ticker', values='volume').fillna(0)
    close_df = raw_df.pivot(index='time', columns='ticker', values='close').ffill().fillna(0)
    open_df = raw_df.pivot(index='time', columns='ticker', values='open').ffill().fillna(0)
    low_df = raw_df.pivot(index='time', columns='ticker', values='low').ffill().fillna(0)
    high_df = raw_df.pivot(index='time', columns='ticker', values='high').ffill().fillna(0)
    
    # -------------------------------------
    # CÁC CHỈ BÁO VĨ MÔ & KỸ THUẬT (DÀNH CHO AI HỌC)
    # -------------------------------------
    market_return_5d = returns_df.rolling(5).sum().mean(axis=1).fillna(0) 
    market_return_20d = returns_df.rolling(20).sum().mean(axis=1).fillna(0) 
    market_vol_20d = returns_df.rolling(20).std().mean(axis=1).fillna(0) 
    
    price_df = np.exp(returns_df.cumsum())
    ma20_df = price_df.rolling(20).mean()
    dist_ma20_df = ((price_df - ma20_df) / ma20_df).fillna(0)

    # -------------------------------------
    # CÁC CHỈ BÁO KỊCH BẢN LUẬT (KHÔNG CHO AI HỌC)
    # -------------------------------------
    ema_df_20 = close_df.ewm(span=20, adjust=False).mean()
    ema_df_50 = close_df.ewm(span=50, adjust=False).mean()
    ema_df_200 = close_df.ewm(span=200, adjust=False).mean()

    # SMA Vol
    def calc_sma(df, n=5):
        return df.rolling(n).mean()
    sma_vol_5_df = calc_sma(vol_df)
    sma_vol_20_df = calc_sma(vol_df, n=20)
    # RSI (Chuẩn Wilder's Smoothing)
    change_df = close_df - close_df.shift(1)
    gain_df = change_df.clip(lower=0)
    loss_df = change_df.clip(upper=0).abs()
    ema_gain = gain_df.ewm(alpha=1/20, min_periods=20, adjust=False).mean()
    ema_loss = loss_df.ewm(alpha=1/20, min_periods=20, adjust=False).mean()
    rs_df = ema_gain / ema_loss
    rsi_df = 100 - (100 / (1 + rs_df))
    rsi_df = rsi_df.fillna(100)

    # MACD
    macd_df = close_df.ewm(span=12, adjust=False).mean() - close_df.ewm(span=26, adjust=False).mean()
    signal_df = macd_df.ewm(span=9, adjust=False).mean()
    hist_df = macd_df - signal_df
    
    # Độ dốc xu hướng (Slope)
    def calc_slope(df, n=3):
        return (df - df.shift(n)) / n
    slope_ema20_3_df = calc_slope(ema_df_20, n=3)  
    slope_sma5_3_df = calc_sma(sma_vol_5_df, n=3)
    # Candlestick Patterns (Đã sửa lỗi Boolean Error)
    body_df = (close_df - open_df).abs()
    tail_df = (np.minimum(close_df, open_df) - low_df)
    head_df = high_df - np.maximum(close_df, open_df)
    
    is_hammer_df = (tail_df >= 2 * body_df) & (head_df <= 0.1 * body_df) & (body_df > 0)
    is_bull_engulf_df = (close_df.shift(1) < open_df.shift(1)) & (close_df > open_df) & (close_df >= open_df.shift(1)) & (open_df <= close_df.shift(1))

    # LowestLow(5) và HighestHigh(10)
    lowest_low_5_df = low_df.rolling(5).min()
    highest_high_5_df = high_df.rolling(5).max()
    highest_high_10_df = high_df.rolling(10).max()
    highest_high_20_df = high_df.rolling(20).max()

    # Đáy và đỉnh
    # Ngày T-2 là Đỉnh nếu nó cao hơn T-4, T-3 (Quá khứ) VÀ cao hơn T-1, T (Hiện tại)
    is_peak = (high_df.shift(2) > high_df.shift(4)) & \
                (high_df.shift(2) > high_df.shift(3)) & \
                (high_df.shift(2) > high_df.shift(1)) & \
                (high_df.shift(2) > high_df)

    # Ngày T-2 là Đáy nếu nó thấp hơn T-4, T-3 (Quá khứ) VÀ thấp hơn T-1, T (Hiện tại)
    is_valley = (low_df.shift(2) < low_df.shift(4)) & \
                (low_df.shift(2) < low_df.shift(3)) & \
                (low_df.shift(2) < low_df.shift(1)) & \
                (low_df.shift(2) < low_df)
    
    peak_values = high_df.shift(2).where(is_peak, np.nan)
    valley_values = low_df.shift(2).where(is_valley, np.nan)

    last_peak_df = peak_values.ffill()
    last_valley_df = valley_values.ffill()
        
    # FIBONACCI
    wave_amplitude_df = last_peak_df - last_valley_df
    wave_state_df = pd.DataFrame(np.nan, index=high_df.index, columns=high_df.columns)
    
    wave_state_df[is_peak] = 1
    wave_state_df[is_valley] = -1
    
    last_swing_type_df = wave_state_df.ffill()
    
    # SUP() RES()
    n = 20 # Số phiên để xác nhận Đỉnh/Đáy (Bạn có thể tinh chỉnh 10 hoặc 20)
    window = 2 * n + 1
    
    # 1. Tìm mức cao nhất/thấp nhất trong 41 ngày gần nhất (20 ngày trước + 1 ngày giữa + 20 ngày sau)
    rolling_max = high_df.rolling(window).max()
    rolling_min = low_df.rolling(window).min()
    
    # 2. XÁC NHẬN ĐỈNH/ĐÁY KHÔNG DÙNG CENTER=TRUE
    # Nếu giá cao nhất của 20 ngày trước (shift(20)) BẰNG với mức Max của toàn bộ 41 ngày
    # -> Chứng tỏ 20 ngày trước là một ĐỈNH CỤC BỘ (Peak)
    is_peak = (high_df.shift(n) == rolling_max)
    is_valley = (low_df.shift(n) == rolling_min)
    
    # 3. Ghi nhớ mức giá tại Đỉnh/Đáy đó
    peak_values = high_df.shift(n).where(is_peak, np.nan)
    valley_values = low_df.shift(n).where(is_valley, np.nan)
    
    # 4. Kéo dài giá trị ra để lấy Hỗ trợ (Sup) và Kháng cự (Res) gần nhất
    res_df = peak_values.ffill() # Đỉnh gần nhất (Res)
    sup_df = valley_values.ffill() # Đáy gần nhất (Sup)


    # SÓNG TĂNG CHÍNH = Mốc cuối cùng được xác nhận là 1 cái Đỉnh
    # Trả về giá trị True/False cho từng ngày
    is_upward_wave_df = (last_swing_type_df == 1)
    fib_382_df = last_peak_df - (0.382 * wave_amplitude_df)
    fib_500_df = last_peak_df - (0.500 * wave_amplitude_df)
    fib_618_df = last_peak_df - (0.618 * wave_amplitude_df)
    fib_ext_127_df = last_peak_df + (0.272 * wave_amplitude_df)
    fib_ext_161_df = last_peak_df + (0.618 * wave_amplitude_df)

     # Bollinger Bands 
    bb_middle_df = close_df.rolling(20).mean()
    bb_std_df = close_df.rolling(20).std()
    bb_upper_df = bb_middle_df + (2 * bb_std_df)
    bb_lower_df = bb_middle_df - (2 * bb_std_df)
    
    # 2. TÍNH ĐỘ RỘNG DẢI BĂNG (BB WIDTH)
    bb_width_df = (bb_upper_df - bb_lower_df) / bb_middle_df
    
    # 3. TÍN HIỆU BIẾN ĐỘNG MẠNH (DẢI BĂNG MỞ RỘNG)
    # So sánh độ rộng hôm nay với hôm qua VÀ với Trung bình 20 ngày
    bb_expanding_df = (bb_width_df > bb_width_df.shift(1)) & \
                        (bb_width_df > bb_width_df.rolling(20).mean())
    
    # ACCUMULATION
    highest_high_21_df = high_df.rolling(21).max().shift(1)
    lowest_low_21_df = low_df.rolling(21).min().shift(1)
        
    # Biên độ dao động của cái hộp nhỏ hơn 15%
    is_accumulation_df = (highest_high_21_df - lowest_low_21_df) / lowest_low_21_df <= 0.15
    # Bóc tách dữ liệu HMM từ raw_df
    prob0_df = raw_df.pivot(index='time', columns='ticker', values='prob_ticker_0').fillna(0) 
    prob1_df = raw_df.pivot(index='time', columns='ticker', values='prob_ticker_1').fillna(0) 
    prob2_df = raw_df.pivot(index='time', columns='ticker', values='prob_ticker_2').fillna(0) 
    vol_20d_df = raw_df.pivot(index='time', columns='ticker', values='rolling_vol_20d').fillna(0)
    ret_20d_df = raw_df.pivot(index='time', columns='ticker', values='return_20d').fillna(0)
    vol_ratio_df = raw_df.pivot(index='time', columns='ticker', values='volume_ratio').fillna(0)

    # Broadcast vĩ mô
    mkt_ret5_df = pd.DataFrame(np.tile(market_return_5d.values.reshape(-1, 1), (1, weights_dim)), index=returns_df.index, columns=returns_df.columns)
    mkt_ret20_df = pd.DataFrame(np.tile(market_return_20d.values.reshape(-1, 1), (1, weights_dim)), index=returns_df.index, columns=returns_df.columns)
    mkt_vol_df = pd.DataFrame(np.tile(market_vol_20d.values.reshape(-1, 1), (1, weights_dim)), index=returns_df.index, columns=returns_df.columns)
    
    # -------------------------------------------------------------------------
    # GOM THÀNH 2 BẢNG BIỆT LẬP (1 CHO AI, 1 CHO LUẬT KỊCH BẢN)
    # -------------------------------------------------------------------------
    # BẢNG 1: Dành cho Mạng Nơ-ron (Chỉ 10 tính năng cốt lõi)
    ai_features_df = pd.concat([
        prob0_df, prob1_df, prob2_df, vol_20d_df, ret_20d_df, vol_ratio_df, 
        mkt_ret5_df, mkt_ret20_df, mkt_vol_df, dist_ma20_df
    ], axis=1).fillna(0)
                                
    # BẢNG 2: Kho vũ khí cho kịch bản của con người (AI không được thấy)

    list_strategies = [vol_df ,low_df ,close_df, 
                       ema_df_20, ema_df_50, ema_df_200, 
                       bb_lower_df, bb_middle_df ,bb_upper_df,
                       rsi_df, hist_df, macd_df,
                       sma_vol_20_df,
                       slope_ema20_3_df, slope_sma5_3_df, 
                       is_hammer_df.astype(float), is_bull_engulf_df.astype(float),
                       lowest_low_5_df, highest_high_5_df ,highest_high_10_df, highest_high_20_df,
                       fib_382_df, fib_500_df, fib_618_df, is_upward_wave_df, fib_ext_127_df, fib_ext_161_df,
                       sup_df, res_df,
                       bb_expanding_df,
                       is_accumulation_df,
                       ]

    strategies_features_df = pd.concat(list_strategies, axis=1).fillna(0)
    
    num_strategies_features = 31

    # Đồng bộ độ dài
    returns_df, ai_features_df = returns_df.align(ai_features_df, axis=0, join='inner')
    returns_df, strategies_features_df = returns_df.align(strategies_features_df, axis=0, join='inner')
    tickers = returns_df.columns.tolist()
    
    return returns_df, ai_features_df, strategies_features_df, weights_dim, tickers, num_strategies_features

## LUẬT 1, 3, 5, 6: MÔI TRƯỜNG ĐẦU TƯ (GYM ENVIRONMENT)

## Lớp này mô phỏng lại Sàn chứng khoán. Nơi AI sẽ thử nghiệm các lệnh Mua/Bán và nhận Phạt/Thưởng (Reward).

In [ ]:
class AdvancedPortfolioEnv(gym.Env):
        def __init__(self, returns_df, ai_features_df, script_features_df, weights_dim, num_script_features, tickers=None, step_size=CONFIG.HOLDING_PERIOD, cost_rate=CONFIG.
  COST_RATE, is_test=False):
            super().__init__()
            self.returns_arr = np.exp(returns_df.values) - 1.0 
            
            # Lưu 2 mảng tách biệt
            self.features_arr = ai_features_df.values
            self.script_arr = script_features_df.values
            
            self.weights_dim = weights_dim
            self.num_script_features = num_script_features
            self.n_steps = len(self.returns_arr)
            
            self.step_size = step_size
            self.cost_rate = cost_rate
            self.is_test = is_test 
            self.tickers = tickers
            
            self.action_space = spaces.Box(low=0, high=1, shape=(self.weights_dim,), dtype=np.float32)
            # Báo cho AI biết là nó chỉ cần học 10 tính năng (vì ai_features_df có 10 cột)
            self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.weights_dim, 10), dtype=np.float32)
            
            self.current_step = 0
            self.weights = np.zeros(self.weights_dim)
            self.entry_prices = np.zeros(self.weights_dim)
            self.entry_scenarios = np.zeros(self.weights_dim)
            self.entry_stages = np.zeros(self.weights_dim)
        def reset(self, seed=None, options=None):
            self.current_step = 0
            self.weights = np.zeros(self.weights_dim)
            return self._get_obs(), {}
            
        def _get_obs(self):
            idx = min(self.current_step, self.n_steps - 1)
            obs_1d = self.features_arr[idx]
            obs_2d = obs_1d.reshape(10, self.weights_dim).T
            return obs_2d.astype(np.float32)
            
        def step(self, action):
            current_obs = self._get_obs()
            idx = min(self.current_step, self.n_steps - 1)
            
            # Bóc tách kho vũ khí kịch bản tại ngày hôm nay
            current_scripts = self.script_arr[idx].reshape(self.num_script_features, self.weights_dim).T
            
            # Dữ liệu hôm qua (t - 1)
            prev_idx = max(0, idx - 1)
            prev_scripts = self.script_arr[prev_idx].reshape(self.num_script_features, self.weights_dim).T
            # ==============================================================
            # KIỂM DUYỆT VÀ GHI ĐÈ LỆNH CỦA AI BẰNG KỊCH BẢN CỦA CON NGƯỜI
            # ==============================================================
            for i in range(self.weights_dim):
                current_regime = np.argmax([current_obs[i][0], current_obs[i][1], current_obs[i][2]]) # 0: Bear, 1: SideWay, 2: Bull
                
                # Trích xuất toàn bộ 21 chỉ báo kịch bản
                volume          = current_scripts[i][0]
                low             = current_scripts[i][1]
                close           = current_scripts[i][2]
                ema20           = current_scripts[i][3]
                ema50           = current_scripts[i][4]
                ema200          = current_scripts[i][5]
                bb_lower        = current_scripts[i][6]
                bb_middle       = current_scripts[i][7]
                bb_upper        = current_scripts[i][8]  # Đã sửa từ 7 thành 8
                rsi             = current_scripts[i][9]  # Tăng lên 1 đơn vị từ đây về sau
                macd_hist       = current_scripts[i][10]
                macd_line       = current_scripts[i][11]
                sma_vol_20      = current_scripts[i][12]
                slope_ema20_3   = current_scripts[i][13]
                slope_sma5_3    = current_scripts[i][14]
                is_hammer       = current_scripts[i][15]
                is_bull_engulf  = current_scripts[i][16]
                lowest_low_5    = current_scripts[i][17]
                highest_high_5  = current_scripts[i][18]
                highest_high_10 = current_scripts[i][19]
                highest_high_20 = current_scripts[i][20]
                fib_382         = current_scripts[i][21]
                fib_500         = current_scripts[i][22]
                fib_618         = current_scripts[i][23]
                is_upward_wave  = current_scripts[i][24]
                fib_ext_127     = current_scripts[i][25]
                fib_ext_161     = current_scripts[i][26]
                sup             = current_scripts[i][27]
                res             = current_scripts[i][28]
                bb_expanding    = current_scripts[i][29]
                is_accumulation = current_scripts[i][30]

                prev_volume     = prev_scripts[i][0]
                prev_close      = prev_scripts[i][1]
                prev_ema20      = prev_scripts[i][3]
                prev_ema50      = prev_scripts[i][4]
                prev_macd_hist  = prev_scripts[i][10]
                prev_macd_line  = prev_scripts[i][11]

                # =======================================================
                # CỖ MÁY TRẠNG THÁI: TÁCH RIÊNG TỪNG KỊCH BẢN NHƯ YÊU CẦU
                # =======================================================
                if self.weights[i] > 0: 
                    # --- ĐÃ CÓ HÀNG -> RẼ NHÁNH TÙY THEO KỊCH BẢN LÚC MUA ---
                    scenario_id = self.entry_scenarios[i]
                    entry_price = self.entry_prices[i]
                    
                    if scenario_id == 1:
                        # ---------------------------------------------------
                        # LUẬT EXIT CỦA KỊCH BẢN 1: EMA PULLBACK
                        # ---------------------------------------------------
                        risk = entry_price - ema50 if entry_price > ema50 else (entry_price * 0.05)
                        stage = self.entry_stages[i]
                        if stage == 1:
                            is_green_confirm = (close > prev_close) and (close > ema20)
                            # ĐIỀU KIỆN 2: Bị úp sọt thủng EMA50 với Vol lớn
                            is_broken_with_vol = (close < ema50) and (volume > sma_vol_20)
                            if is_broken_with_vol:
                                # [XẤU]: Hủy deal gia tăng. Giữ nguyên 30% nằm im chờ T+2.5 cắt lỗ
                                action[i] = self.weights[i]
                            elif is_green_confirm:
                                # [ĐẸP]: Quất nốt 70% còn lại!
                                action[i] = max(action[i], 0.7)
                                self.entry_stages[i] = 2 # Đánh dấu đã Full hàng
                            else:
                                # [SIDEWAY]: Nến lình xình, chưa có dấu hiệu rõ ràng -> Nằm chờ tiếp
                                action[i] = self.weights[i]
                        elif stage == 2:
                                # ĐÃ FULL 100% HÀNG -> ÁP DỤNG EXIT LINE ĐỘNG ĐỂ CHỐT LỜI/CẮT LỖ
                                # (Đường MACD cắt xuống hoặc EMA20 cắt xuống EMA50)
                            is_macd_dead_cross = (macd_hist < 0) and (prev_macd_hist >= 0)
                            is_ema_dead_cross = (ema20 < ema50) and (prev_ema20 >= prev_ema50)
                            if close < ema50 or close < (lowest_low_5 * 0.98): 
                                action[i] = 0.0
                                self.entry_scenarios[i] = 0
                            elif close >= highest_high_10: 
                                action[i] = 0.0
                                self.entry_scenarios[i] = 0
                            elif close >= (entry_price + 2 * risk): 
                                action[i] = 0.0
                                self.entry_scenarios[i] = 0
                            elif close < ema20: 
                                action[i] = 0.0
                                self.entry_scenarios[i] = 0
                            else:
                                action[i] = max(action[i], self.weights[i])
                            
                    elif scenario_id == 2:
                        # ---------------------------------------------------
                        # LUẬT EXIT & GIẢI NGÂN (NHỒI LỆNH) CỦA KỊCH BẢN 2: FIBONACCI
                        # ---------------------------------------------------
                        stage = self.entry_stages[i] # 1: Thăm dò 30%, 2: Full 100%
                        is_green_candle = close > current_scripts[i][2] # Dùng tạm close thay đổi (hoặc close > open nếu có)
                        
                        if stage == 1:
                            if close < fib_618: # Xấu: Thủng đáy Fib
                                action[i] = self.weights[i] # Ôm chặt chờ T+2.5 cắt
                            elif is_green_candle and close > entry_price: # Đẹp: Nến xanh xác nhận
                                action[i] = max(action[i], 0.2) # Gia tăng 70% còn lại
                                self.entry_stages[i] = 2
                            else:
                                action[i] = self.weights[i]
                                
                        elif stage == 2:
                            # Đã full hàng -> Canh chốt lời/cắt lỗ
                            if close < fib_618:
                                action[i] = 0.0
                                self.entry_scenarios[i] = 0
                                self.entry_stages[i] = 0
                            elif close >= fib_ext_161 or close >= highest_high_10:
                                action[i] = 0.0
                                self.entry_scenarios[i] = 0
                                self.entry_stages[i] = 0
                            else:
                                action[i] = max(action[i], self.weights[i])
                    elif scenario_id == 3:
                        if close < sup:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                        elif close >= highest_high_10 or close >= ema50:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                    elif scenario_id == 4:
                        if close < ema50:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                        elif close >= highest_high_20:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                    elif scenario_id == 5:
                        if close < bb_middle:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                    elif scenario_id == 6:
                        if close < res * 0.97:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                        elif close >= fib_ext_127 or close >= fib_ext_161 or close >= (entry_price - res* 0.97) / (min(fib_ext_127, fib_ext_161) - entry_price):
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                    elif scenario_id == 7:
                        if close < lowest_low_5:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                        elif close >= highest_high_5:
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                    elif scenario_id == 8:
                        if (macd_hist < 0 and prev_macd_line >= 0) or (ema20 < ema50 and prev_ema20 >= prev_ema50):
                            action[i] = 0.0
                            self.entry_scenarios[i] = 0
                            self.entry_prices[i] = 0.0
                    else:
                        # Fallback an toàn nếu lỗi
                        action[i] = 0.0
                        self.entry_scenarios[i] = 0

                else:
                    # ---------------------------------------------------
                    # LỚP 1: TRẠNG THÁI 2 (CHƯA CÓ HÀNG) -> CHUẨN BỊ MUA
                    # ---------------------------------------------------
                    is_bought = False
                    
                    # ---------------------------------------------------
                    # LỚP 2: KIỂM TRA ĐIỀU KIỆN  (TICKER REGIME)
                    # ---------------------------------------------------
                    if current_regime == 2: 
                        # ===============================================
                        # THỊ TRƯỜNG BULL
                        # ===============================================
                        
                        # EMA PULLBACK
                        if not is_bought and (ema20 > ema50) and (close > ema50) and (slope_ema20_3 > 0):
                            if (low <= ema20) and (40 <= rsi <= 55) and (is_bull_engulf == 1.0 or is_hammer == 1.0) and (volume > prev_volume):
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 1 
                                    self.entry_stages[i] = 1    
                                    action[i] = action[i]
                                    is_bought = True
                                    
                        # FIBONACCI PULLBACK 
                        if not is_bought and is_upward_wave == 1.0:
                            if (fib_618 <= close <= fib_382) and (is_hammer == 1.0 or is_bull_engulf == 1.0):
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 2 
                                    self.entry_stages[i] = 1    
                                    action[i] = action[i]
                                    is_bought = True
                        # SUPPORT BOUNCE
                        if not is_bought and low <= sup:
                            if volume < sma_vol_20 and is_bull_engulf == 1.0:
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 3
                                    action[i] = action[i] * 0.7
                                    is_bought = True
                        # RSI OVER SOLD IN BULL
                        if not is_bought and ema20 > ema50:
                            if 30 <= rsi <= 40 and low <= ema50 and is_hammer == 1.0:
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 4
                                    action[i] = action[i] * 0.5
                                    is_bought = True
                        # BOLLING TREND RIDE 
                        if not is_bought and bb_expanding == 1.0:
                            if close >= (bb_upper * 0.99) and slope_ema20_3 > 0 and volume > prev_volume:
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 5
                                    action[i] = action[i] * 0.5
                                    is_bought = True
                        # BREAKOUT RESISTANCE
                        if not is_bought and is_accumulation == 1.0:
                            if close >= res and volume > (1.5 * sma_vol_20) and macd_hist > 0: 
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 6
                                    action[i] = action[i] * 0.5
                                    is_bought = True
                        # MACD ZERO LINE BOUNCE 
                        if not is_bought and (-0.1 <= macd_line <= 0.1) and macd_hist > 0 and prev_macd_line <= 0:
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 7
                                    action[i] = action[i] * 0.3
                                    is_bought = True    
                        # GOLDEN CROSS
                        if not is_bought and close > ema200:
                            if ema20 > ema50 and prev_ema20 <= prev_ema50 and macd_line > 0:
                                if action[i] > 0.0:
                                    self.entry_prices[i] = close
                                    self.entry_scenarios[i] = 8
                                    action[i] = action[i] * 0.3
                                    is_bought = True

                    elif current_regime == 1:
                        # ===============================================
                        # BỘ CHIÊU THỨC DÀNH CHO THỊ TRƯỜNG SIDEWAY (ĐI NGANG)
                        # ===============================================
                        pass # Bạn có thể thêm kịch bản Đánh 2 biên Bollinger Bands vào đây
                        
                    elif current_regime == 0:
                        # ===============================================
                        # THỊ TRƯỜNG BEAR (DOWN TREND)
                        # ===============================================
                        pass # Đứng ngoài ôm tiền mặt, tuyệt đối không mua
                        
                    # NẾU KHÔNG CÓ KỊCH BẢN NÀO THỎA MÃN
                    if not is_bought:
                        action[i] = 0.0
            
            conviction = np.max(action)
            investment_ratio = conviction / CONFIG.CASH_CONVICTION_THRESHOLD if conviction < CONFIG.CASH_CONVICTION_THRESHOLD else 1.0 
                
            action = action / (np.sum(action) + 1e-9)
            action = action * investment_ratio 
                
            cost = self.cost_rate * np.sum(np.abs(action - self.weights))
            self.weights = action
            
            end_step = min(self.current_step + self.step_size, self.n_steps)
            days_held = end_step - self.current_step
            
            if days_held == 0:
                return self._get_obs(), 0, True, False, {}
                
            daily_portfolio_returns = []
            cum_ret_since_rebalance = 0.0
            is_cash_mode = False
            
            for t in range(self.current_step, end_step):
                if is_cash_mode:
                    daily_portfolio_returns.append(0.0)
                    continue
                    
                daily_ret = np.sum(self.weights * self.returns_arr[t])
                if t == self.current_step:
                    daily_ret -= cost
                    
                daily_portfolio_returns.append(daily_ret)
                cum_ret_since_rebalance = (1 + cum_ret_since_rebalance) * (1 + daily_ret) - 1
                days_since_buy = t - self.current_step
                
                # Ngắt mạch cắt lỗ chốt lời theo T+2
                if days_since_buy >= CONFIG.T_PLUS_SETTLEMENT:
                    if cum_ret_since_rebalance <= CONFIG.STOP_LOSS_THRESHOLD or cum_ret_since_rebalance >= CONFIG.TAKE_PROFIT_THRESHOLD:
                        exit_cost = self.cost_rate * np.sum(self.weights)
                        daily_portfolio_returns[-1] -= exit_cost 
                        self.weights = np.zeros(self.weights_dim) 
                        is_cash_mode = True 
            
            daily_portfolio_returns = np.array(daily_portfolio_returns)
            cum_return = np.prod(1 + daily_portfolio_returns) - 1
            
            mean_ret = np.mean(daily_portfolio_returns)
            std_ret = np.std(daily_portfolio_returns) + 1e-9
            sharpe = (mean_ret / std_ret) * np.sqrt(252) 
            
            reward = sharpe + CONFIG.PENALTY_FOR_LOSS if cum_return < 0 else sharpe
                
            self.current_step = end_step
            done = self.current_step >= self.n_steps - 1 
            
            return self._get_obs(), float(reward), done, False, {'net_return': cum_return}

## MẠNG NƠ-RON TRÍ TUỆ NHÂN TẠO (NEURAL NETWORK STRUCTURE)

## Đây là "Bộ Não" thực sự của AI. Mạng Nơ-ron này chịu trách nhiệm Đọc dữ liệu (Extractor).

In [123]:
class AdvancedTickerExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        num_tickers = observation_space.shape[0] # 46 mã
        num_features_per_ticker = observation_space.shape[1] # 11 tính năng
        
        # Tầng 1 (Local): Học phân tích RIÊNG LẺ từng mã cổ phiếu (Ticker-level)
        self.ticker_net = nn.Sequential(
            nn.Linear(num_features_per_ticker, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        # Tầng 2 (Global): Tổng hợp toàn bộ 46 mã lại để ra quyết định Toàn cục
        self.global_net = nn.Sequential(
            nn.Linear(num_tickers * 16, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: th.Tensor) -> th.Tensor:
        batch_size, num_tickers, num_features = observations.shape
        # Nhồi 11 tính năng vào Tầng 1
        obs_reshaped = observations.view(batch_size * num_tickers, num_features)
        ticker_features = self.ticker_net(obs_reshaped)
        
        # Đưa Tầng 1 vào Tầng 2
        ticker_features = ticker_features.view(batch_size, num_tickers * 16)
        global_features = self.global_net(ticker_features)
        return global_features

In [124]:
def walk_forward_train_test(returns_df, ai_features_df, strategies_features_df, weights_dim, tickers, num_strategies_features):
    print("\n[LUẬT 4] KHỞI ĐỘNG WALK-FORWARD VALIDATION (CHỐNG HỌC VẸT)...")
    
    total_days = len(returns_df)
    chunk_size = total_days // CONFIG.WALK_FORWARD_SPLITS
    
    # Chia dữ liệu: Trong Fold 1, Học từ Khúc 0 đến Khúc 2, Thi đấu Khúc 2 đến Cuối cùng
    folds = [
        (0, chunk_size * 2, chunk_size * 2, total_days) 
    ]
    
    # Gắn Não (Neural Network) vào Trái Tim (PPO Algorithm)
    policy_kwargs = dict(
        features_extractor_class=AdvancedTickerExtractor,
        features_extractor_kwargs=dict(features_dim=256),
    )
    
    for fold_idx, (train_start, train_end, test_start, test_end) in enumerate(folds):
        print(f"\n=======================================================")
        print(f"--- FOLD {fold_idx + 1}: HỌC TỪ NGÀY {train_start}->{train_end}, THI ĐẤU NGÀY {test_start}->{test_end} ---")
        
        # Dữ liệu Train
        returns_train = returns_df.iloc[train_start:train_end]
        ai_train = ai_features_df.iloc[train_start:train_end]
        strategies_train = strategies_features_df.iloc[train_start:train_end]
        
        # Dữ liệu Test (Tương lai chưa biết)
        returns_test = returns_df.iloc[test_start:test_end]
        ai_test = ai_features_df.iloc[test_start:test_end]
        strategies_test = strategies_features_df.iloc[test_start:test_end]
        
        # Tạo Sàn Chứng Khoán Giả lập cho quá trình Học
        train_env = DummyVecEnv([lambda: AdvancedPortfolioEnv(
            returns_train, 
            ai_train, 
            strategies_train, 
            weights_dim, 
            num_strategies_features, 
            tickers=tickers
        )])
        
        # Khởi tạo Mô hình Trí tuệ Nhân tạo - Sử dụng Giải thuật PPO (Thuật toán số 1 hiện nay cho Trading)
        model = PPO("MlpPolicy", train_env, 
                    policy_kwargs=policy_kwargs, 
                    verbose=1, # Bật hiển thị Log lúc Học để theo dõi sức khỏe AI (loss, entropy...)
                    n_steps=10000,
                    learning_rate=0.00005,
                    ent_coef=CONFIG.ENT_COEF,
                    batch_size=90)
                    
        print(f"Đang huấn luyện mô hình ({CONFIG.TRAINING_TIMESTEPS} steps)...")
        # Kích hoạt quá trình Học tập
        model.learn(total_timesteps=CONFIG.TRAINING_TIMESTEPS)
        
        print(f"Đang chạy Backtest thực chiến (CÓ TÍNH PHÍ GIAO DỊCH {CONFIG.COST_RATE * 100}%)...")
        print("-------------------------------------------------------")
        
        # Tạo Sàn Chứng Khoán Thật cho quá trình Thi Đấu (is_test=True để in Log)
        test_env = DummyVecEnv([lambda: AdvancedPortfolioEnv(
            returns_test, 
            ai_test, 
            strategies_test, 
            weights_dim, 
            num_strategies_features, 
            tickers=tickers, 
            is_test=True
        )])
        obs = test_env.reset()
        
        portfolio_returns = []
        done = [False]
        
        # Vòng lặp Thi đấu: Từ Ngày đầu tiên của Tương lai đến Ngày cuối cùng
        while not done[0]:
            # AI tự nhìn Biểu đồ và Quyết định Xuống tiền
            action, _ = model.predict(obs, deterministic=True)
            # Hệ thống ghi nhận Quyết định, trừ Tiền Phí và Phản hồi Kết Quả (Lãi/Lỗ)
            obs, reward, done, info = test_env.step(action)
            
            # Ghi chép Lợi nhuận
            if 'net_return' in info[0]:
                portfolio_returns.append(info[0]['net_return'])
            
        # Tổng kết Kì thi
        test_len = len(returns_test)
        cum_ret = (np.prod(1 + np.array(portfolio_returns)) - 1) * 100
        print(f"=> [KẾT QUẢ] LỢI NHUẬN TÍCH LŨY FOLD {fold_idx + 1} (SAU {test_len} NGÀY THỰC CHIẾN): {cum_ret:.2f}%")

if __name__ == "__main__":
    returns_df, ai_features_df, strategies_features_df, weights_dim, tickers, num_strategies_features = load_data()
    walk_forward_train_test(returns_df, ai_features_df, strategies_features_df, weights_dim, tickers, num_strategies_features)



[LUẬT 4] KHỞI ĐỘNG WALK-FORWARD VALIDATION (CHỐNG HỌC VẸT)...

--- FOLD 1: HỌC TỪ NGÀY 0->1588, THI ĐẤU NGÀY 1588->2382 ---
Using cpu device
Đang huấn luyện mô hình (500000 steps)...
------------------------------
| time/              |       |
|    fps             | 834   |
|    iterations      | 1     |
|    time_elapsed    | 11    |
|    total_timesteps | 10000 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 674         |
|    iterations           | 2           |
|    time_elapsed         | 29          |
|    total_timesteps      | 20000       |
| train/                  |             |
|    approx_kl            | 0.006957903 |
|    clip_fraction        | 0.0656      |
|    clip_range           | 0.2         |
|    entropy_loss         | -65.3       |
|    explained_variance   | -0.00311    |
|    learning_rate        | 5e-05       |
|    loss                 | 86.5        |
|    n_upda